# Etapa 2. Conhecendo os dados de chuva

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/02_dados_chuva.ipynb)

Aqui você conhece os dados com que o trabalho inteiro é feito: **370 pluviômetros do Paraná**, de janeiro de 1981
a julho de 2020. O objetivo é responder: onde estão as estações, qual a qualidade das séries, quanto chove em cada
região, como a chuva se distribui ao longo do ano e em que meses ela é mais irregular.

As figuras e tabelas são gravadas no seu Drive, na pasta `spi_oni_pr/resultados`.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula:
#   1) define DE ONDE os dados serão lidos (direto do GitHub, no Colab);
#   2) conecta o seu Google Drive e cria a pasta spi_oni_pr/resultados, onde as figuras e
#      tabelas ficam guardadas. O Colab apaga tudo quando a sessão fecha; o Drive, não.
# Na primeira vez, o Colab pede autorização para acessar o Drive: aceite.
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if Path("../dados").exists():
    DADOS = "../dados/"                         # computador do orientador, dentro da pasta do projeto
else:
    DADOS = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/dados/"

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = Path("/content/drive/MyDrive/spi_oni_pr/resultados")
else:
    SAIDA = Path("../resultados")

SAIDA.mkdir(parents=True, exist_ok=True)        # cria a pasta, se ainda não existir
print("Dados lidos de:", DADOS)
print("Resultados gravados em:", SAIDA)

In [ ]:
import pandas as pd
import geopandas as gpd          # pandas com mapas (já vem instalado no Colab)

# Aparência padrão dos gráficos do trabalho (rode uma vez; vale para todos os gráficos do notebook)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,       # tira a borda de cima
    "axes.spines.right": False,     # tira a borda da direita
    "axes.grid": True,              # grade de fundo...
    "grid.color": "#e6e5e0",        # ...bem clara, para não competir com os dados
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,         # grade atrás das barras e linhas
})

AZUL = "#2a78d6"                    # cor única quando há só uma série
# uma cor fixa para cada estação em destaque (a cor acompanha a estação em todas as figuras)
CORES = {"Santa Helena": "#2a78d6", "Cascavel": "#eb6834", "Foz do Iguaçu": "#1baf7a", "Mercedes": "#eda100"}

## 1. As estações

`estacoes_pr.csv` tem uma linha por estação: código, município, mesorregião do IBGE, latitude, longitude,
altitude, início e fim da série, a porcentagem de dias preenchidos e o nome das estações em destaque.

In [ ]:
est = pd.read_csv(DADOS + "estacoes_pr.csv", dtype={"cod": str, "cod_ibge": str})

print("Número de estações:", len(est))
est.head()

`value_counts()` conta quantas vezes cada valor aparece numa coluna: aqui, quantas estações há em cada
mesorregião.

In [ ]:
por_meso = est["mesorregiao"].value_counts()
por_meso

As quatro estações em destaque são as que têm algo escrito na coluna `destaque`. `.notna()` quer dizer "não está
vazio".

In [ ]:
destaque = est[est["destaque"].notna()]
destaque[["cod", "municipio", "mesorregiao", "lat", "lon", "alt_m", "inicio", "fim", "pct_preenchido"]]

## 2. Mapa das estações

O mapa usa os limites das mesorregiões do IBGE (`pr_mesorregioes.geojson`). Cada estação é um ponto azul; as
quatro em destaque aparecem em preto, com o nome. O nome de cada mesorregião é escrito no centro dela, por isso não
é preciso uma cor para cada região.

In [ ]:
meso = gpd.read_file(DADOS + "pr_mesorregioes.geojson")

fig, ax = plt.subplots(figsize=(9, 6.5))
meso.plot(ax=ax, color="#f4f3ef", edgecolor="#9a9890", linewidth=0.8)    # mesorregiões: fundo claro

# todas as estações
ax.scatter(est["lon"], est["lat"], s=10, color=AZUL, label="Estação pluviométrica (370)")

# estações em destaque, uma por uma, com o nome ao lado
for i, linha in destaque.iterrows():                 # iterrows percorre a tabela linha a linha
    ax.scatter(linha["lon"], linha["lat"], s=40, color="black", marker="^")
    ax.annotate(linha["municipio"], (linha["lon"], linha["lat"]), xytext=(4, 4),
                textcoords="offset points", fontsize=8)

# nome de cada mesorregião no centro dela
for i, linha in meso.iterrows():
    centro = linha["geometry"].representative_point()
    ax.annotate(linha["mesorregiao"].replace(" Paranaense", ""), (centro.x, centro.y),
                ha="center", fontsize=7, color="#52514e")

ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_aspect("equal")
ax.grid(False)
ax.legend(loc="lower left", fontsize=8)
fig.savefig(SAIDA / "fig01_mapa_estacoes.png", dpi=300, bbox_inches="tight")   # grava no Drive
plt.show()

## 3. Qualidade das séries

Cada estação teve algumas falhas (dias sem leitura) preenchidas com a grade de Xavier et al. (2022). Só entraram no
estudo as estações com no máximo 10% de dias preenchidos. `describe()` resume a coluna: número de valores, média,
desvio padrão, mínimo, quartis e máximo.

In [ ]:
est["pct_preenchido"].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(est["pct_preenchido"], bins=20, color=AZUL, edgecolor="white")   # histograma: quantas estações em cada faixa
ax.set_xlabel("Dias preenchidos (%)")
ax.set_ylabel("Número de estações")
ax.set_title("Porcentagem de dias preenchidos por estação")
plt.show()

## 4. Chuva anual

`chuva_mensal_pr.csv` tem a chuva de cada mês de cada estação. Para a chuva **anual**, somamos os 12 meses de cada
ano. Usamos só os anos completos (1981 a 2019): 2020 vai só até julho.

Primeiro, do jeito simples, para **uma estação** (Santa Helena), com um laço pelos anos.

In [ ]:
mensal = pd.read_csv(DADOS + "chuva_mensal_pr.csv", dtype={"cod": str})
print("Linhas:", len(mensal))

sh = mensal[mensal["cod"] == "2454012"]          # Santa Helena

anos = []
totais = []
for ano in range(1981, 2020):                    # 1981, 1982, ..., 2019
    meses_do_ano = sh[sh["ano"] == ano]
    anos.append(ano)
    totais.append(meses_do_ano["chuva_mm"].sum())

media_sh = sum(totais) / len(totais)
print("Chuva média anual em Santa Helena:", round(media_sh), "mm")
print("Ano mais seco:", anos[totais.index(min(totais))], "com", round(min(totais)), "mm")
print("Ano mais chuvoso:", anos[totais.index(max(totais))], "com", round(max(totais)), "mm")

Para as 370 estações, o laço ficaria longo. O `groupby` faz a mesma conta para todas de uma vez: junta as linhas de
mesma estação e mesmo ano e soma. Depois, tiramos a média dos anos de cada estação.

In [ ]:
completos = mensal[mensal["ano"] <= 2019]                                   # só anos completos

anual = completos.groupby(["cod", "ano"])["chuva_mm"].sum().reset_index()   # chuva de cada ano, por estação
media_anual = anual.groupby("cod")["chuva_mm"].mean().reset_index()         # média dos anos, por estação
media_anual = media_anual.rename(columns={"chuva_mm": "chuva_anual_mm"})

est = est.merge(media_anual, on="cod")       # merge junta as duas tabelas pela coluna cod
print("Santa Helena pelo groupby:", round(est.loc[est["cod"] == "2454012", "chuva_anual_mm"].iloc[0]), "mm")

Resumo por mesorregião: número de estações e chuva anual média, mínima e máxima entre as estações. Esta é a
**Tabela 2** do TCC; ela também é gravada no Drive.

In [ ]:
tabela2 = est.groupby("mesorregiao")["chuva_anual_mm"].agg(["count", "mean", "min", "max"]).round(0)
tabela2.columns = ["estacoes", "media_mm", "minimo_mm", "maximo_mm"]
tabela2 = tabela2.sort_values("media_mm", ascending=False)
tabela2.to_csv(SAIDA / "tabela02_chuva_anual_mesorregiao.csv")
tabela2

O mapa da chuva anual usa **uma só cor**, do claro (pouca chuva) ao escuro (muita chuva). Cores diferentes para
cada faixa (arco-íris) enganam o olho: parecem saltos onde não há.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

azuis = LinearSegmentedColormap.from_list("azuis", ["#cde2fb", "#6da7ec", "#2a78d6", "#184f95", "#0d366b"])

fig, ax = plt.subplots(figsize=(9, 6.5))
meso.plot(ax=ax, color="#f4f3ef", edgecolor="#9a9890", linewidth=0.8)
pontos = ax.scatter(est["lon"], est["lat"], c=est["chuva_anual_mm"], cmap=azuis, s=18,
                    edgecolor="white", linewidth=0.3)
fig.colorbar(pontos, ax=ax, label="Chuva média anual (mm), 1981–2019", shrink=0.7)
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_aspect("equal")
ax.grid(False)
fig.savefig(SAIDA / "fig02_mapa_chuva_anual.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Regime mensal (normal climatológica 1981–2010)

A **normal climatológica** é a média de 30 anos. Para cada estação, calculamos a chuva média de cada mês de 1981 a
2010; depois, a média das estações de cada mesorregião. Um painel por mesorregião, todos com a mesma escala, para
poder comparar.

In [ ]:
normal = mensal[(mensal["ano"] >= 1981) & (mensal["ano"] <= 2010)]         # & = "e"
normal_est = normal.groupby(["cod", "mes"])["chuva_mm"].mean().reset_index()  # média de cada mês, por estação
normal_est = normal_est.merge(est[["cod", "mesorregiao", "municipio"]], on="cod")
normal_meso = normal_est.groupby(["mesorregiao", "mes"])["chuva_mm"].mean().reset_index()
normal_meso.head(12)

In [ ]:
nomes_meses = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
mesorregioes = list(tabela2.index)            # na ordem da mais chuvosa para a menos chuvosa

fig, eixos = plt.subplots(2, 5, figsize=(14, 5.5), sharey=True)   # 2 linhas x 5 colunas, mesmo eixo y
for i, nome in enumerate(mesorregioes):       # enumerate dá o número (i) e o nome de cada item
    ax = eixos.flat[i]
    dados = normal_meso[normal_meso["mesorregiao"] == nome]
    ax.bar(dados["mes"], dados["chuva_mm"], color=AZUL, width=0.7)   # x = número do mês (1 a 12)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(nomes_meses)                                 # e a letra do mês embaixo de cada barra
    ax.set_title(nome.replace(" Paranaense", ""), fontsize=9)
eixos[0, 0].set_ylabel("Chuva (mm/mês)")
eixos[1, 0].set_ylabel("Chuva (mm/mês)")
fig.suptitle("Chuva média mensal por mesorregião, 1981–2010")
fig.tight_layout()
fig.savefig(SAIDA / "fig03_regime_mensal_mesorregioes.png", dpi=300, bbox_inches="tight")
plt.show()

O mesmo para as quatro estações em destaque, uma linha por estação. Cada estação tem sempre a mesma cor em todas as
figuras do trabalho (dicionário `CORES`).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i, linha in destaque.iterrows():
    nome = linha["municipio"]
    dados = normal_est[normal_est["cod"] == linha["cod"]]
    ax.plot(dados["mes"], dados["chuva_mm"], color=CORES[nome], linewidth=2, marker="o", markersize=4, label=nome)
    ultimo = dados.iloc[-1]                                     # rótulo no fim de cada linha
    ax.annotate(nome, (12, ultimo["chuva_mm"]), xytext=(5, 0), textcoords="offset points",
                fontsize=8, va="center")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(nomes_meses)
ax.set_xlim(0.6, 13.6)
ax.set_ylabel("Chuva (mm/mês)")
ax.set_title("Chuva média mensal nas estações em destaque, 1981–2010")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, frameon=False)  # legenda abaixo do gráfico
fig.savefig(SAIDA / "fig04_regime_mensal_destaque.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Variabilidade: coeficiente de variação

O **coeficiente de variação** (CV) mede o quanto a chuva de um mês muda de um ano para outro:
CV = desvio padrão ÷ média × 100. CV de 30% quer dizer que, num ano típico, a chuva daquele mês fica uns 30% acima
ou abaixo da média. Quanto maior o CV, mais arriscado é contar com a chuva daquele mês.

In [ ]:
estat = completos.groupby(["cod", "mes"])["chuva_mm"].agg(["mean", "std"]).reset_index()
estat["cv"] = estat["std"] / estat["mean"] * 100
estat = estat.merge(est[["cod", "mesorregiao"]], on="cod")

cv_meso = estat.groupby(["mesorregiao", "mes"])["cv"].mean().unstack().round(0)   # unstack: meses viram colunas
cv_meso.columns = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
cv_meso.to_csv(SAIDA / "tabela03_cv_mensal_mesorregiao.csv")
cv_meso

---

## O que observar

- O **Sudoeste** e o **Oeste** chovem mais que o **Norte**; os valores mais altos do estado estão no litoral
  (dentro da mesorregião Metropolitana de Curitiba). Compare com Aparecido et al. (2016).
- O **Norte** e o **Noroeste** têm inverno (junho a agosto) bem menos chuvoso; no **Sul**, a chuva é mais bem
  distribuída ao longo do ano.
- O CV é **maior no inverno**: é quando a chuva é mais irregular.

## Exercícios

**Exercício 1.** Qual a estação mais chuvosa e a menos chuvosa do Paraná? Em que municípios ficam? (Dica:
`est.sort_values("chuva_anual_mm")`.)

In [ ]:
# escreva sua resposta aqui

**Exercício 2.** Na figura das estações em destaque, qual é o mês mais chuvoso e o menos chuvoso de cada estação?
As quatro têm o mesmo regime?

*Escreva sua resposta aqui.*

**Exercício 3.** Olhando a tabela do CV, em que meses a chuva do Oeste é mais irregular? O que isso significa para
a semeadura da soja (setembro a novembro) e para o milho segunda safra (fevereiro a junho)?

*Escreva sua resposta aqui.*

---

**Entregar.** As figuras `fig01` a `fig04`, as tabelas `tabela02` e `tabela03` (estão no seu Drive) e os três
exercícios.

**O que escrever.** No TCC2: em *Material e métodos*, conferir o item 4.1 (os números de estações por mesorregião
estão na tabela do item 1 deste notebook) e inserir a Figura 1 (`fig01`). Em *Resultados e discussão*, o item 5.1,
seguindo as orientações em azul.